In [22]:
import os
import glob
import pandas as pd
import pickle
import matplotlib.pyplot as plt
import numpy as np
import random
from datetime import datetime, timedelta
from dateutil.relativedelta import relativedelta
import pprint
import pyspark
import pyspark.sql.functions as F

from pyspark.sql.functions import col
from pyspark.sql.types import StringType, IntegerType, FloatType, DateType

from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import make_scorer, roc_auc_score, fbeta_score

import xgboost as xgb

In [23]:
spark = pyspark.sql.SparkSession.builder \
    .appName("dev") \
    .master("local[*]") \
    .getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

In [24]:
pd.set_option('display.max_columns', None)

In [25]:
print(f'\n{"="*60}')
print('MODEL TRAINING — EXPERIMENTS')
print(f'{"="*60}')

print('\n--- Load labels ---')
label_folder_path = "datamart/gold/label_store/gold_label_store_*"
label_store_sdf = spark.read.parquet(label_folder_path)
print(f'Loaded from: {label_folder_path}')
print('row_count:', label_store_sdf.count())
label_store_sdf = label_store_sdf.withColumnRenamed('snapshot_date', 'mob6_snapshot_date')


MODEL TRAINING — EXPERIMENTS

--- Load labels ---
Loaded from: datamart/gold/label_store/gold_label_store_*
row_count: 12500


In [26]:
print('\n--- Load features ---')
feature_folder_path = "datamart/gold/feature_store/gold_joined.parquet"
feature_store_sdf = spark.read.parquet(feature_folder_path)
print(f'Loaded from: {feature_folder_path}')
print('row_count:', feature_store_sdf.count())


--- Load features ---
Loaded from: datamart/gold/feature_store/gold_joined.parquet
row_count: 12500


In [27]:
xy_sdf = feature_store_sdf.join(label_store_sdf, on='Customer_ID', how='outer')
xyclean_sdf = xy_sdf.drop('label_def', 'loan_id', 'mob6_snapshot_date')
print("Joined and cleaned dataset ready")

Joined and cleaned dataset ready


In [28]:
print('\n--- Split config ---')
model_train_date_str = "2024-09-01"
train_test_period_months = 15
oot_period_months = 5
train_test_ratio = 0.8

config = {}
config["model_train_date_str"] = model_train_date_str
config["train_test_period_months"] = train_test_period_months
config["oot_period_months"] = oot_period_months
config["model_train_date"] = datetime.strptime(model_train_date_str, "%Y-%m-%d")
config["oot_end_date"] = config['model_train_date'] - timedelta(days=1)
config["oot_start_date"] = config['model_train_date'] - relativedelta(months=oot_period_months)
config["train_test_end_date"] = config["oot_start_date"] - timedelta(days=1)
config["train_test_start_date"] = config["oot_start_date"] - relativedelta(months=train_test_period_months)
config["train_test_ratio"] = train_test_ratio

pprint.pprint(config)


--- Split config ---
{'model_train_date': datetime.datetime(2024, 9, 1, 0, 0),
 'model_train_date_str': '2024-09-01',
 'oot_end_date': datetime.datetime(2024, 8, 31, 0, 0),
 'oot_period_months': 5,
 'oot_start_date': datetime.datetime(2024, 4, 1, 0, 0),
 'train_test_end_date': datetime.datetime(2024, 3, 31, 0, 0),
 'train_test_period_months': 15,
 'train_test_ratio': 0.8,
 'train_test_start_date': datetime.datetime(2023, 1, 1, 0, 0)}


In [29]:
oot_sdf = xyclean_sdf.filter(
    (col('snapshot_date') >= config['oot_start_date']) &
    (col('snapshot_date') <= config['oot_end_date'])
)
oot_pdf = oot_sdf.toPandas()
print('OOT row counts:')
print(oot_pdf['snapshot_date'].value_counts().sort_index())

traintest_sdf = xyclean_sdf.filter(
    (col('snapshot_date') <= config['train_test_end_date']) &
    (col('snapshot_date') >= config['train_test_start_date'])
)
traintest_pdf = traintest_sdf.toPandas()

OOT row counts:
snapshot_date
2024-04-01    513
2024-05-01    491
2024-06-01    498
2024-07-01    505
2024-08-01    543
Name: count, dtype: int64


In [30]:
for df_ in [traintest_pdf, oot_pdf]:
    df_["snapshot_date"] = pd.to_datetime(df_["snapshot_date"])
    df_["snapshot_year"] = df_["snapshot_date"].dt.year
    df_["snapshot_month"] = df_["snapshot_date"].dt.month
    df_["snapshot_quarter"] = df_["snapshot_date"].dt.quarter

exclude_cols = ['Customer_ID', 'label', 'snapshot_date']
feature_cols = [c for c in traintest_pdf.columns if c not in exclude_cols]
fe_cols = [f'fe_{i}_mean' for i in range(1, 21)]
print(f'Feature columns ({len(feature_cols)} total)')

Feature columns (60 total)


In [31]:
X_temp = traintest_pdf[feature_cols].copy()
y_temp = traintest_pdf['label'].copy()
X_oot = oot_pdf[feature_cols].copy()
y_oot = oot_pdf['label'].copy()

X_train, X_test, y_train, y_test = train_test_split(
    X_temp, y_temp, test_size=0.2, random_state=55, stratify=y_temp
)

for name, y in [("train", y_train), ("test", y_test), ("oot", y_oot)]:
    print(f"{name:6} | rows: {len(y):>7,} | bad rate: {y.mean():.4f}")

train  | rows:   5,977 | bad rate: 0.2871
test   | rows:   1,495 | bad rate: 0.2870
oot    | rows:   2,550 | bad rate: 0.2996


## Common Preprocessing (shared across all models)

In [32]:
# mean imputation
mean_cols = ['Num_Bank_Accounts', 'Num_Credit_Card', 'Num_of_Loan',
             'Num_of_Delayed_Payment', 'Age']
mean_impute_vals = {}
for c in mean_cols:
    mean_val = X_train[c].mean()
    mean_impute_vals[c] = mean_val
    X_train[c] = X_train[c].fillna(mean_val)
    X_test[c] = X_test[c].fillna(mean_val)
    X_oot[c] = X_oot[c].fillna(mean_val)

# median imputation
median_cols = ['Delay_from_due_date', 'Num_Credit_Inquiries',
               'Monthly_Balance', 'Interest_Rate']
median_impute_vals = {}
for c in median_cols:
    median_val = X_train[c].median()
    median_impute_vals[c] = median_val
    X_train[c] = X_train[c].fillna(median_val)
    X_test[c] = X_test[c].fillna(median_val)
    X_oot[c] = X_oot[c].fillna(median_val)

# OHE
cat_cols = ['Occupation', 'Credit_Mix', 'Payment_of_Min_Amount',
            'Spending_Behaviour', 'Payments_Size']
ohe = OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore')
ohe.fit(X_train[cat_cols])
for X in [X_train, X_test, X_oot]:
    encoded = ohe.transform(X[cat_cols])
    encoded_df = pd.DataFrame(encoded, columns=ohe.get_feature_names_out(cat_cols), index=X.index)
    X.drop(columns=cat_cols, inplace=True)
    X[encoded_df.columns] = encoded_df

print(f'Mean imputation: {mean_cols}')
print(f'Median imputation: {median_cols}')
print(f'OHE: {cat_cols}')
print(f'Columns after OHE: {len(X_train.columns)}')

Mean imputation: ['Num_Bank_Accounts', 'Num_Credit_Card', 'Num_of_Loan', 'Num_of_Delayed_Payment', 'Age']
Median imputation: ['Delay_from_due_date', 'Num_Credit_Inquiries', 'Monthly_Balance', 'Interest_Rate']
OHE: ['Occupation', 'Credit_Mix', 'Payment_of_Min_Amount', 'Spending_Behaviour', 'Payments_Size']
Columns after OHE: 80


In [33]:
# Save base copies before fe_ imputation / scaling
X_train_base = X_train.copy()
X_test_base = X_test.copy()
X_oot_base = X_oot.copy()

scale_cols = [
    'Age', 'Annual_Income', 'Monthly_Inhand_Salary',
    'Interest_Rate', 'Outstanding_Debt', 'Total_EMI_per_month',
    'Amount_invested_monthly', 'Monthly_Balance',
    'Changed_Credit_Limit', 'Credit_Utilization_Ratio',
    'Credit_History_Age_Years', 'snapshot_year',
    'Num_Bank_Accounts', 'Num_Credit_Card', 'Num_of_Loan',
    'Num_Credit_Inquiries', 'Num_of_Delayed_Payment',
    'Delay_from_due_date', 'clickstream_days',
    'EMI_to_Salary_Ratio', 'Debt_to_Income', 'Disposable_Income',
    'months12_to_annual_income_ratio',
    'Loan_Auto_Loan', 'Loan_Credit_Builder_Loan',
    'Loan_Debt_Consolidation_Loan', 'Loan_Home_Equity_Loan',
    'Loan_Mortgage_Loan', 'Loan_Not_Specified', 'Loan_Payday_Loan',
    'Loan_Personal_Loan', 'Loan_Student_Loan'
] + fe_cols

auc_scorer = make_scorer(roc_auc_score)


## Model 1: XGBoost — NaN fe_ (native missing value handling)

In [34]:
X_train_v2 = X_train_base.copy()
X_test_v2  = X_test_base.copy()
X_oot_v2   = X_oot_base.copy()

# fe_ columns left as NaN - Exclude fe_ from scaling since StandardScaler cannot handle NaN
scale_cols_v2 = [c for c in scale_cols if c not in fe_cols]

scaler_v2 = StandardScaler()
scaler_v2.fit(X_train_v2[scale_cols_v2])
X_train_v2[scale_cols_v2] = scaler_v2.transform(X_train_v2[scale_cols_v2])
X_test_v2[scale_cols_v2]  = scaler_v2.transform(X_test_v2[scale_cols_v2])
X_oot_v2[scale_cols_v2]   = scaler_v2.transform(X_oot_v2[scale_cols_v2])
print('fe_ columns left as NaN, scaling applied to all other columns')

fe_ columns left as NaN, scaling applied to all other columns


In [35]:
param_dist = {
    'n_estimators': [100, 200, 300],
    'max_depth': [2, 3],
    'learning_rate': [0.01, 0.05, 0.1],
    'subsample': [0.7, 0.8],
    'colsample_bytree': [0.7, 0.8],
    'gamma': [0, 0.1, 0.5],
    'min_child_weight': [5, 10, 20],
    'reg_lambda': [10, 20, 50]
}

xgb_clf_v2 = xgb.XGBClassifier(eval_metric='logloss', scale_pos_weight=3, random_state=55)

random_search_v2 = RandomizedSearchCV(
    estimator=xgb_clf_v2,
    param_distributions=param_dist,
    scoring=auc_scorer,
    n_iter=50,
    cv=5,
    verbose=1,
    random_state=55,
    n_jobs=-1
)
random_search_v2.fit(X_train_v2, y_train)

best_xgb_v2 = random_search_v2.best_estimator_
train_auc_v2 = roc_auc_score(y_train, best_xgb_v2.predict_proba(X_train_v2)[:, 1])
test_auc_v2 = roc_auc_score(y_test,  best_xgb_v2.predict_proba(X_test_v2)[:, 1])
oot_auc_v2 = roc_auc_score(y_oot,   best_xgb_v2.predict_proba(X_oot_v2)[:, 1])

train_f2_v2 = fbeta_score(y_train, best_xgb_v2.predict(X_train_v2), beta=2)
test_f2_v2 = fbeta_score(y_test,  best_xgb_v2.predict(X_test_v2),  beta=2)
oot_f2_v2 = fbeta_score(y_oot,   best_xgb_v2.predict(X_oot_v2),   beta=2)

print('\nXGB')
print(f'Best params: {random_search_v2.best_params_}')
print(f'Train AUC: {train_auc_v2:.4f} | Gini: {round(2*train_auc_v2-1, 3)} | F2: {train_f2_v2:.4f}')
print(f'Test  AUC: {test_auc_v2:.4f} | Gini: {round(2*test_auc_v2-1, 3)} | F2: {test_f2_v2:.4f}')
print(f'OOT   AUC: {oot_auc_v2:.4f} | Gini: {round(2*oot_auc_v2-1, 3)} | F2: {oot_f2_v2:.4f}')

Fitting 5 folds for each of 50 candidates, totalling 250 fits

XGB
Best params: {'subsample': 0.7, 'reg_lambda': 20, 'n_estimators': 300, 'min_child_weight': 10, 'max_depth': 3, 'learning_rate': 0.05, 'gamma': 0.1, 'colsample_bytree': 0.7}
Train AUC: 0.9306 | Gini: 0.861 | F2: 0.8326
Test  AUC: 0.9001 | Gini: 0.8 | F2: 0.7907
OOT   AUC: 0.8871 | Gini: 0.774 | F2: 0.7748


## Model 2: Logistic Regression

In [36]:
X_train_v3 = X_train_base.copy()
X_test_v3 = X_test_base.copy()
X_oot_v3 = X_oot_base.copy()

# apply fe imputation
fe_impute_vals_v3 = {}
for c in fe_cols:
    mean_val = X_train_v3.loc[X_train_v3['has_clickstream'] == 1, c].mean()
    fe_impute_vals_v3[c] = mean_val
    X_train_v3[c] = X_train_v3[c].fillna(mean_val)
    X_test_v3[c] = X_test_v3[c].fillna(mean_val)
    X_oot_v3[c] = X_oot_v3[c].fillna(mean_val)

scaler_v3 = StandardScaler()
scaler_v3.fit(X_train_v3[scale_cols])
X_train_v3[scale_cols] = scaler_v3.transform(X_train_v3[scale_cols])
X_test_v3[scale_cols]  = scaler_v3.transform(X_test_v3[scale_cols])
X_oot_v3[scale_cols]   = scaler_v3.transform(X_oot_v3[scale_cols])
print('V3: fe_ mean imputation + full scaling applied for LR')

V3: fe_ mean imputation + full scaling applied for LR


In [37]:
lr_clf = LogisticRegression(random_state=55, class_weight='balanced', max_iter=1000)

param_dist_lr = {
    'C': [0.001, 0.01, 0.1, 1, 10, 100],
    'penalty': ['l1', 'l2'],
    'solver': ['liblinear']
}

random_search_v3 = RandomizedSearchCV(
    estimator=lr_clf,
    param_distributions=param_dist_lr,
    scoring=auc_scorer,
    n_iter=12,
    cv=5,
    verbose=1,
    random_state=55,
    n_jobs=-1
)
random_search_v3.fit(X_train_v3, y_train)

best_lr_v3 = random_search_v3.best_estimator_
train_auc_v3 = roc_auc_score(y_train, best_lr_v3.predict_proba(X_train_v3)[:, 1])
test_auc_v3  = roc_auc_score(y_test,  best_lr_v3.predict_proba(X_test_v3)[:, 1])
oot_auc_v3   = roc_auc_score(y_oot,   best_lr_v3.predict_proba(X_oot_v3)[:, 1])

train_f2_v3 = fbeta_score(y_train, best_lr_v3.predict(X_train_v3), beta=2)
test_f2_v3  = fbeta_score(y_test,  best_lr_v3.predict(X_test_v3),  beta=2)
oot_f2_v3   = fbeta_score(y_oot,   best_lr_v3.predict(X_oot_v3),   beta=2)

print('\nogistic Regression')
print(f'Best params: {random_search_v3.best_params_}')
print(f'Train AUC: {train_auc_v3:.4f} | Gini: {round(2*train_auc_v3-1, 3)} | F2: {train_f2_v3:.4f}')
print(f'Test  AUC: {test_auc_v3:.4f} | Gini: {round(2*test_auc_v3-1, 3)} | F2: {test_f2_v3:.4f}')
print(f'OOT   AUC: {oot_auc_v3:.4f} | Gini: {round(2*oot_auc_v3-1, 3)} | F2: {oot_f2_v3:.4f}')

Fitting 5 folds for each of 12 candidates, totalling 60 fits

ogistic Regression
Best params: {'solver': 'liblinear', 'penalty': 'l1', 'C': 10}
Train AUC: 0.8546 | Gini: 0.709 | F2: 0.7279
Test  AUC: 0.8710 | Gini: 0.742 | F2: 0.7572
OOT   AUC: 0.8108 | Gini: 0.622 | F2: 0.5983


## Results Comparison

In [38]:
results = pd.DataFrame({
    'Model': ['XGBoost', 'Logistic Regression'],
    'Train AUC':[train_auc_v2, train_auc_v3],
    'Test AUC': [test_auc_v2,  test_auc_v3],
    'OOT AUC': [oot_auc_v2,   oot_auc_v3],
    'Train Gini':[round(2*train_auc_v2-1, 3), round(2*train_auc_v3-1, 3)],
    'Test Gini': [round(2*test_auc_v2-1, 3), round(2*test_auc_v3-1, 3)],
    'OOT Gini':[round(2*oot_auc_v2-1, 3), round(2*oot_auc_v3-1, 3)],
    'Train F2': [round(train_f2_v2, 4), round(train_f2_v3, 4)],
    'Test F2': [round(test_f2_v2, 4),round(test_f2_v3, 4)],
    'OOT F2': [round(oot_f2_v2, 4), round(oot_f2_v3, 4)]
})
results

,Model,Train AUC,Test AUC,OOT AUC,Train Gini,Test Gini,OOT Gini,Train F2,Test F2,OOT F2
0,XGBoost,0.930605,0.900147,0.887064,0.861,0.800,0.774,0.8326,0.7907,0.7748
1,Logistic Regression,0.854622,0.870964,0.810846,0.709,0.742,0.622,0.7279,0.7572,0.5983
